# ERA5 Temperature Inversion — Data Crawling

**Tầng đảo nhiệt (Temperature Inversion):** Khi không khí ấm nằm trên không khí lạnh → ô nhiễm bị nhốt gần mặt đất → PM2.5 tăng cao.

**Các lớp áp suất cần:**
- `1000 hPa` — gần mặt đất (~100m)
- `925 hPa` — ~750m
- `850 hPa` — ~1500m

**Features tính ra:**
- `t_inv_850_1000` = T(850) − T(1000): dương → có tầng đảo nhiệt mạnh
- `t_inv_925_1000` = T(925) − T(1000): đảo nhiệt tầng thấp
- `t_inv_850_925`  = T(850) − T(925): đảo nhiệt tầng giữa
- `t_1000`, `t_925`, `t_850`: nhiệt độ từng tầng

**Product:** `reanalysis-era5-pressure-levels`  
**Credentials:** Cùng CDS API key với ERA5 BLH

## Cell 1 — Setup

In [9]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
pip("cdsapi"); pip("xarray"); pip("netCDF4")
print("Ready.")

Ready.


## Cell 2 — Cấu hình

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd

CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "e07e7cb2-ac0a-4459-92c0-c97f21e9af35"
(Path.home() / ".cdsapirc").write_text(f"url: {CDS_URL}\nkey: {CDS_KEY}\n")

ROOT     = Path("D:/Bussiness_plan/Multimodal_PM25")
RAW_DIR  = ROOT / "data/raw/era5_tinv"; RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR = ROOT / "data/processed";     PROC_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV  = PROC_DIR / "era5_t_inversion_daily.csv"
MERGED   = PROC_DIR / "daily_merged.csv"

STATIONS = {
    2161292: (21.0152, 105.7999),
    2161306: (21.0500, 105.7400),
    4946811: (21.0491, 105.8831),
    4946812: (21.0031, 105.7947),
    4946813: (21.0052, 105.8418),
    6123215: (20.9933, 105.9441),
}

LAT_N = max(v[0] for v in STATIONS.values()) + 0.5
LAT_S = min(v[0] for v in STATIONS.values()) - 0.5
LON_W = min(v[1] for v in STATIONS.values()) - 0.5
LON_E = max(v[1] for v in STATIONS.values()) + 0.5

PRESSURE_LEVELS = ["850", "925", "1000"]  # hPa

YEAR_CONFIGS = [
    {"year": "2024", "months": [f"{m:02d}" for m in range(1, 13)]},
    {"year": "2025", "months": [f"{m:02d}" for m in range(1, 13)]},
    {"year": "2026", "months": [f"{m:02d}" for m in range(1, 6)]},
]

print(f"Bbox: N={LAT_N} W={LON_W} S={LAT_S} E={LON_E}")
print(f"Pressure levels: {PRESSURE_LEVELS} hPa")

Bbox: N=21.55 W=105.24 S=20.4933 E=106.4441
Pressure levels: ['850', '925', '1000'] hPa


In [15]:
import cdsapi

days  = [f"{d:02d}" for d in range(1, 32)]
hours = [f"{h:02d}:00" for h in range(0, 24)]

# Tung thang rieng biet - an toan nhat (~1-2 MB/thang)
MONTHS_TO_DOWNLOAD = (
    [(2024, m) for m in range(1, 13)] +
    [(2025, m) for m in range(1, 13)] +
    [(2026, m) for m in range(1, 6)]
)

c = cdsapi.Client()
downloaded = []

for (yr, mo) in MONTHS_TO_DOWNLOAD:
    tag  = f"{yr}_{mo:02d}"
    part = RAW_DIR / f"era5_tinv_{tag}.nc"
    downloaded.append(part)

    if part.exists():
        print(f"  [{tag}] skip ({part.stat().st_size/1e6:.1f} MB)")
        continue

    print(f"  [{tag}] downloading ...", end=" ", flush=True)
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type":   "reanalysis",
            "variable":       "temperature",
            "pressure_level": PRESSURE_LEVELS,
            "year":           str(yr),
            "month":          f"{mo:02d}",
            "day":            days,
            "time":           hours,
            "area":           [LAT_N, LON_W, LAT_S, LON_E],
            "format":         "netcdf",
        },
        str(part),
    )
    print(f"done ({part.stat().st_size/1e6:.1f} MB)")

# Summary
ok      = sum(1 for p in downloaded if p.exists())
missing = [p.name for p in downloaded if not p.exists()]
print(f"\nDownloaded: {ok}/{len(downloaded)}")
if missing:
    print(f"Missing   : {missing}")

  [2024_01] downloading ... 

2026-05-20 17:22:28,378 INFO Request ID is 14ac98cb-e4fd-443b-88d5-4c31c2ba3a07
2026-05-20 17:22:30,181 INFO status has been updated to accepted
2026-05-20 17:22:48,123 INFO status has been updated to running
2026-05-20 17:25:28,628 INFO status has been updated to successful
                                                                                      

done (0.2 MB)
  [2024_02] downloading ... 

2026-05-20 17:25:33,131 INFO Request ID is c2d7d67a-d4c7-4e61-8b10-c9d006a965a9
2026-05-20 17:25:33,452 INFO status has been updated to accepted
2026-05-20 17:25:43,014 INFO status has been updated to running
2026-05-20 17:31:57,634 INFO status has been updated to successful
                                                                                       

done (0.2 MB)
  [2024_03] downloading ... 

...

done (0.2 MB)
  [2026_04] downloading ... 

2026-05-20 19:01:47,895 INFO Request ID is f2ee12ea-56f5-4cd2-a2be-76b028aa1a26
2026-05-20 19:01:48,405 INFO status has been updated to accepted
2026-05-20 19:01:57,712 INFO status has been updated to running
2026-05-20 19:06:12,161 INFO status has been updated to successful
                                                                                      

done (0.2 MB)
  [2026_05] downloading ... 

2026-05-20 19:06:18,361 INFO Request ID is d6fe2958-7e3e-4041-8a0d-f1551698b8ae
2026-05-20 19:06:18,714 INFO status has been updated to accepted
2026-05-20 19:06:28,029 INFO status has been updated to running
2026-05-20 19:06:41,969 INFO status has been updated to accepted
2026-05-20 19:06:53,626 INFO status has been updated to running
2026-05-20 19:08:15,652 INFO status has been updated to successful
                                                                                       

done (0.1 MB)

Downloaded: 29/29


In [16]:
import xarray as xr

all_parts = sorted(RAW_DIR.glob("era5_tinv_????_??.nc"))
print(f"Files found: {len(all_parts)}")
for p in all_parts[:3]:   # show first 3
    ds  = xr.open_dataset(p)
    tc  = "valid_time" if "valid_time" in ds.coords else "time"
    t   = pd.to_datetime(ds[tc].values)
    lev = next((c for c in ["pressure_level","level","plev"] if c in ds.coords), None)
    print(f"  {p.name}: dims={dict(ds.dims)}  levels={ds[lev].values if lev else '?'}"
          f"  steps={len(t)}")
    ds.close()
if len(all_parts) > 3:
    print(f"  ... and {len(all_parts)-3} more files")

Files found: 29
  era5_tinv_2024_01.nc: dims={'valid_time': 744, 'pressure_level': 3, 'latitude': 5, 'longitude': 5}  levels=[1000.  925.  850.]  steps=744
  era5_tinv_2024_02.nc: dims={'valid_time': 696, 'pressure_level': 3, 'latitude': 5, 'longitude': 5}  levels=[1000.  925.  850.]  steps=696
  era5_tinv_2024_03.nc: dims={'valid_time': 744, 'pressure_level': 3, 'latitude': 5, 'longitude': 5}  levels=[1000.  925.  850.]  steps=744
  ... and 26 more files


C:\Users\Admin\AppData\Local\Temp\ipykernel_25396\3856900529.py:10: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  {p.name}: dims={dict(ds.dims)}  levels={ds[lev].values if lev else '?'}"
C:\Users\Admin\AppData\Local\Temp\ipykernel_25396\3856900529.py:10: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  {p.name}: dims={dict(ds.dims)}  levels={ds[lev].values if lev else '?'}"
C:\Users\Admin\AppData\Local\Temp\ipykernel_25396\3856900529.py:10: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more con

In [17]:
def nearest_idx(arr, target):
    return int(np.argmin(np.abs(arr - target)))

def process_nc(nc_path):
    ds   = xr.open_dataset(nc_path)
    lats = ds["latitude"].values
    lons = ds["longitude"].values
    tc   = "valid_time" if "valid_time" in ds.coords else "time"
    var  = next((v for v in ds.data_vars if v.lower() in ("t","temperature","temp")),
                list(ds.data_vars)[0])
    lev_coord = next((c for c in ["pressure_level","level","plev"]
                      if c in ds.coords or c in ds.dims), None)

    da = ds[var]
    for dim in ["expver", "number"]:
        if dim in da.dims:
            da = da.isel({dim: 0})

    times  = pd.to_datetime(da[tc].values)
    levels = da[lev_coord].values.astype(float)

    i850  = nearest_idx(levels, 850)
    i925  = nearest_idx(levels, 925)
    i1000 = nearest_idx(levels, 1000)

    records = []
    for loc_id, (lat, lon) in STATIONS.items():
        il = nearest_idx(lats, lat)
        ij = nearest_idx(lons, lon)

        def get_t(ilev):
            return (da.isel({lev_coord: ilev, "latitude": il, "longitude": ij})
                    .values.flatten().astype(float) - 273.15)

        df_hr = pd.DataFrame({
            "time":  times,
            "t850":  get_t(i850),
            "t925":  get_t(i925),
            "t1000": get_t(i1000),
        })
        df_hr["date"]         = df_hr["time"].dt.normalize()
        df_hr["inv_850_1000"] = df_hr["t850"] - df_hr["t1000"]
        df_hr["inv_925_1000"] = df_hr["t925"] - df_hr["t1000"]
        df_hr["is_morning"]   = df_hr["time"].dt.hour.isin([23, 0, 1, 2])

        agg = (df_hr.groupby("date")
               .agg(t850_mean        =("t850",         "mean"),
                    t925_mean        =("t925",         "mean"),
                    t1000_mean       =("t1000",        "mean"),
                    inv_850_1000_mean=("inv_850_1000", "mean"),
                    inv_850_1000_max =("inv_850_1000", "max"),
                    inv_925_1000_mean=("inv_925_1000", "mean"))
               .reset_index())

        morn = (df_hr[df_hr["is_morning"]]
                .groupby("date")["inv_850_1000"].mean()
                .rename("inv_850_1000_morning").reset_index())
        agg = agg.merge(morn, on="date", how="left")
        agg.insert(0, "location_id", loc_id)
        records.append(agg)

    ds.close()
    return pd.concat(records, ignore_index=True)


# Process all monthly files
all_parts = sorted(RAW_DIR.glob("era5_tinv_????_??.nc"))
if not all_parts:
    print("No files found. Run Cell 3 first.")
else:
    print(f"Processing {len(all_parts)} files ...")
    parts = [process_nc(p) for p in all_parts]
    tinv_df = (pd.concat(parts, ignore_index=True)
               .drop_duplicates(subset=["location_id","date"])
               .sort_values(["location_id","date"])
               .reset_index(drop=True))

    print(f"\nTotal rows : {len(tinv_df)}")
    print(f"Date range : {tinv_df['date'].min()} -> {tinv_df['date'].max()}")
    print(tinv_df[[c for c in tinv_df.columns if "inv" in c]].describe().round(2))

Processing 29 files ...

Total rows : 5196
Date range : 2024-01-01 00:00:00 -> 2026-05-15 00:00:00
       inv_850_1000_mean  inv_850_1000_max  inv_925_1000_mean  \
count            5196.00           5196.00            5196.00   
mean               -6.79             -4.29              -3.78   
std                 2.24              2.52               1.06   
min               -10.98             -9.40              -6.00   
25%                -8.33             -6.06              -4.42   
50%                -7.35             -5.03              -3.95   
75%                -5.61             -2.84              -3.43   
max                 2.70              4.12               2.88   

       inv_850_1000_morning  
count               5196.00  
mean                  -5.57  
std                    2.28  
min                  -10.24  
25%                   -7.18  
50%                   -6.17  
75%                   -4.35  
max                    3.49  


In [18]:
import xarray as xr

for cfg in YEAR_CONFIGS:
    p = RAW_DIR / f"era5_tinv_{cfg['year']}.nc"
    if not p.exists():
        print(f"MISSING: {p.name}"); continue
    ds = xr.open_dataset(p)
    tc = "valid_time" if "valid_time" in ds.coords else "time"
    t  = pd.to_datetime(ds[tc].values)
    print(f"{p.name}:")
    print(f"  dims   = {dict(ds.dims)}")
    print(f"  vars   = {list(ds.data_vars)}")
    print(f"  levels = {ds['pressure_level'].values if 'pressure_level' in ds.coords else ds.coords}")
    print(f"  time   : {t[0].date()} -> {t[-1].date()}  ({len(t)} steps)")
    ds.close()

MISSING: era5_tinv_2024.nc
MISSING: era5_tinv_2025.nc
MISSING: era5_tinv_2026.nc


## Cell 5 — Extract → daily T inversion per station

In [19]:
def nearest_idx(arr, target):
    return int(np.argmin(np.abs(arr - target)))

def process_nc(nc_path):
    ds   = xr.open_dataset(nc_path)
    lats = ds["latitude"].values
    lons = ds["longitude"].values
    tc   = "valid_time" if "valid_time" in ds.coords else "time"

    # Detect temperature variable name
    var = [v for v in ds.data_vars if "t" in v.lower() or "temp" in v.lower()]
    var = var[0] if var else list(ds.data_vars)[0]

    # Detect pressure level coordinate
    lev_coord = None
    for c in ["pressure_level", "level", "plev"]:
        if c in ds.coords or c in ds.dims:
            lev_coord = c; break

    da = ds[var]
    for dim in ["expver", "number"]:
        if dim in da.dims:
            da = da.isel({dim: 0})

    times  = pd.to_datetime(da[tc].values)
    levels = da[lev_coord].values.astype(float) if lev_coord else None
    print(f"  {nc_path.name}: var={var}  levels={levels}  steps={len(times)}")

    def get_level(hpa):
        """Select temperature series at nearest pressure level."""
        il = nearest_idx(levels, hpa) if levels is not None else 0
        return il

    i850  = get_level(850)
    i925  = get_level(925)
    i1000 = get_level(1000)

    records = []
    for loc_id, (lat, lon) in STATIONS.items():
        il = nearest_idx(lats, lat)
        ij = nearest_idx(lons, lon)

        # Extract T at each pressure level (K → °C)
        def get_t(ilev):
            if lev_coord:
                vals = da.isel({lev_coord: ilev, "latitude": il, "longitude": ij})
            else:
                vals = da.isel(latitude=il, longitude=ij)
            arr = vals.values.flatten().astype(float)
            return arr - 273.15  # K to Celsius

        t850  = get_t(i850)
        t925  = get_t(i925)
        t1000 = get_t(i1000)

        df_hr = pd.DataFrame({
            "time":  times,
            "t850":  t850,
            "t925":  t925,
            "t1000": t1000,
        })
        df_hr["date"] = df_hr["time"].dt.normalize()

        # Inversion indices
        df_hr["inv_850_1000"] = df_hr["t850"]  - df_hr["t1000"]
        df_hr["inv_925_1000"] = df_hr["t925"]  - df_hr["t1000"]
        df_hr["inv_850_925"]  = df_hr["t850"]  - df_hr["t925"]

        # Daily stats
        agg_cols = ["t850", "t925", "t1000",
                    "inv_850_1000", "inv_925_1000", "inv_850_925"]
        daily = df_hr.groupby("date")[agg_cols].agg(["mean", "max"]).reset_index()

        # Flatten multi-level columns
        daily.columns = ["date"] + [
            f"{c}_{s}" for c, s in daily.columns[1:]
        ]

        # Key feature: morning inversion (06-10h local = UTC-1 to UTC+3)
        df_hr["is_morning"] = df_hr["time"].dt.hour.isin([23, 0, 1, 2])
        morn = (df_hr[df_hr["is_morning"]]
                .groupby("date")["inv_850_1000"].mean()
                .rename("inv_850_1000_morning").reset_index())
        daily = daily.merge(morn, on="date", how="left")
        daily.insert(0, "location_id", loc_id)
        records.append(daily)

    ds.close()
    return pd.concat(records, ignore_index=True)


nc_parts = [RAW_DIR / f"era5_tinv_{cfg['year']}.nc" for cfg in YEAR_CONFIGS]
existing = [p for p in nc_parts if p.exists()]

parts = [process_nc(p) for p in existing]
tinv_df = (pd.concat(parts, ignore_index=True)
           .drop_duplicates(subset=["location_id", "date"])
           .sort_values(["location_id", "date"])
           .reset_index(drop=True))

print(f"\nTotal rows : {len(tinv_df)}")
print(f"Date range : {tinv_df['date'].min()} -> {tinv_df['date'].max()}")
inv_cols = [c for c in tinv_df.columns if "inv" in c]
print(f"\nInversion columns: {inv_cols}")
print(tinv_df[inv_cols].describe().round(2))

ValueError: No objects to concatenate

## Cell 6 — Visualize T inversion theo mùa

In [ ]:
import matplotlib.pyplot as plt

tinv_df["month"] = pd.to_datetime(tinv_df["date"]).dt.month
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly mean inversion
monthly = tinv_df.groupby("month")[["inv_850_1000_mean", "inv_925_1000_mean"]].mean()
monthly.plot(ax=axes[0], marker="o")
axes[0].axhline(0, color="red", ls="--", lw=1, alpha=0.7, label="inversion threshold")
axes[0].set_title("T Inversion theo thang", fontsize=12)
axes[0].set_xlabel("Thang"); axes[0].set_ylabel("T(upper) - T(1000hPa) [°C]")
axes[0].set_xticks(range(1,13)); axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].text(0.05, 0.95, "Dương = đảo nhiệt = PM2.5 cao",
             transform=axes[0].transAxes, fontsize=9, color="red")

# Correlation with PM2.5
main = pd.read_csv(MERGED, parse_dates=["date"])
tinv_df["date"] = pd.to_datetime(tinv_df["date"])
check = main.merge(tinv_df[["location_id","date","inv_850_1000_mean"]],
                   on=["location_id","date"], how="left")
valid = check[["pm25","inv_850_1000_mean"]].dropna()
r = valid.corr().iloc[0, 1]

axes[1].scatter(valid["inv_850_1000_mean"], valid["pm25"],
                alpha=0.3, s=8, color="steelblue")
axes[1].set_xlabel("T inversion 850-1000 hPa (°C)"); axes[1].set_ylabel("PM2.5")
axes[1].set_title(f"T Inversion vs PM2.5  r={r:.3f}", fontsize=12)
axes[1].axvline(0, color="red", ls="--", lw=1, alpha=0.5)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / "outputs/t_inversion_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Correlation T_inv vs PM2.5: r = {r:.3f}")

## Cell 7 — Lưu CSV + Merge vào daily_merged.csv

In [20]:
tinv_df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  shape={tinv_df.shape}")

main = pd.read_csv(MERGED, parse_dates=["date"])

# Columns to merge
merge_cols = [c for c in tinv_df.columns
              if c not in ["location_id", "date", "month"]]

# Remove old columns if re-running
for col in merge_cols:
    if col in main.columns:
        main.drop(columns=[col], inplace=True)

tinv_df["date"] = pd.to_datetime(tinv_df["date"])
main = main.merge(
    tinv_df[["location_id", "date"] + merge_cols],
    on=["location_id", "date"], how="left"
)

# Coverage check
key_cols = ["inv_850_1000_mean", "inv_925_1000_mean", "inv_850_1000_morning"]
for col in key_cols:
    if col in main.columns:
        pct = main[col].notna().mean() * 100
        print(f"  {col}: {pct:.1f}% coverage")

main.to_csv(MERGED, index=False)
print(f"\nUpdated: {MERGED}  shape={main.shape}")
print(f"T inversion cols: {[c for c in main.columns if 'inv' in c]}")

Saved: D:\Bussiness_plan\Multimodal_PM25\data\processed\era5_t_inversion_daily.csv  shape=(5196, 9)
  inv_850_1000_mean: 100.0% coverage
  inv_925_1000_mean: 100.0% coverage
  inv_850_1000_morning: 100.0% coverage

Updated: D:\Bussiness_plan\Multimodal_PM25\data\processed\daily_merged.csv  shape=(2274, 75)
T inversion cols: ['inv_850_1000_mean', 'inv_850_1000_max', 'inv_925_1000_mean', 'inv_850_1000_morning']


## Cell 8 — Thêm vào train_3d_v2.py

```python
# ctx_feats
ctx_feats = [
    ...,
    "inv_850_1000_mean",    # T inversion mạnh nhất (850-1000 hPa)
    "inv_925_1000_mean",    # T inversion tầng thấp
    "inv_850_1000_morning", # Inversion sáng sớm (quan trọng nhất)
]

# tab_cols (trees)
tab_cols = [
    ...,
    "inv_850_1000_mean", "inv_850_1000_max",
    "inv_925_1000_mean", "inv_850_1000_morning",
    "t_1000_mean", "t_850_mean",  # absolute T at each level
]
```

**Lý do T inversion quan trọng cho đô thị:**
- Hà Nội nằm trong lòng chảo địa hình → inversion giữ ô nhiễm lại
- Mùa đông (tháng 11-2): inversion mạnh nhất, PM2.5 cao nhất
- Sáng sớm: BLH thấp + inversion → rush hour emissions không khuếch tán được